# 02 — Live inference via Groq (free tier)

Runs the engine against a real LLM without any local setup. Groq offers a generous free tier with `llama-3.3-70b-versatile` and `llama-3.1-8b-instant` — no credit card.

**You will learn:**
1. How to route the engine to any OpenAI-compatible endpoint via `OPENAI_BASE_URL`.
2. What real output quality looks like on an 8 B-class model.
3. How the trace reports per-node latency for a live run.

**Prerequisites:** free Groq API key from https://console.groq.com. **Time:** ~3 minutes per query. **Cost:** $0.

## Step 1 — install + secrets

Put your Groq key in Colab's secret manager (the 🔑 icon in the left sidebar) with the name `GROQ_API_KEY`. **Do not paste it into a cell.**

In [ ]:
%%capture
!git clone --depth 1 https://github.com/TheAiSingularity/agentic-research-engine-oss.git /content/engine-repo
!pip install -q langgraph openai rank_bm25 trafilatura pypdf pyyaml

import sys
sys.path.insert(0, '/content/engine-repo')

In [ ]:
import os
from google.colab import userdata

# Route the engine at Groq's OpenAI-compatible endpoint.
os.environ['OPENAI_BASE_URL'] = 'https://api.groq.com/openai/v1'
os.environ['OPENAI_API_KEY']  = userdata.get('GROQ_API_KEY')

# Pick models. llama-3.1-8b-instant is the cheap/fast tier; 70b is slower but stronger.
FAST_MODEL  = 'llama-3.1-8b-instant'
STRONG_MODEL = 'llama-3.3-70b-versatile'

for k in ('MODEL_PLANNER', 'MODEL_SEARCHER', 'MODEL_CRITIC', 'MODEL_ROUTER', 'MODEL_VERIFIER', 'MODEL_COMPRESSOR'):
    os.environ[k] = FAST_MODEL
os.environ['MODEL_SYNTHESIZER'] = STRONG_MODEL

# Groq's streaming is a little flaky for our pattern — batched is more reliable.
os.environ['ENABLE_STREAM'] = '0'

# We don't have SearXNG running on Colab. Skip web search: use evidence-less
# mode where the pipeline passes through. For real web search, Tutorial 03
# shows how to run against a public SearXNG instance.
os.environ['ENABLE_FETCH']  = '0'
os.environ['SEARXNG_URL']   = 'https://searx.be'   # a public SearXNG instance

print('Configured. Models:')
print('  synthesizer:', STRONG_MODEL)
print('  everything else:', FAST_MODEL)

## Step 2 — one live query

Run a factual question through the pipeline. The answer, sources, CoVe-verified claims, and per-node trace all come back together.

In [ ]:
from engine.interfaces.common import run_query, format_sources, format_verified_summary, format_trace_per_node

Q = "What is Anthropic's contextual retrieval, and what percentage reduction in retrieval failures did they report?"

rr = run_query(Q, domain='papers')

In [ ]:
print(f'Q: {Q}')
print()
print(f'A: {rr.answer}')
print()
print('── hallucination check ──')
print(format_verified_summary(rr))
for c in rr.verified_claims:
    print('  ✓', c['text'])
for c in rr.unverified_claims:
    print('  ✗', c)
print()
print('── sources ──')
for row in format_sources(rr):
    mark = '●' if row['fetched'] else '○'
    print(f"  [{row['idx']}] {mark} {row['url']}")
print()
print('── trace (per-node totals) ──')
for row in format_trace_per_node(rr):
    print(f"  {row['node']:12s} calls={row['calls']:2d} latency={row['latency_s']:.2f}s tokens~{row['tokens_est']}")
print()
print(f'total wall: {rr.total_latency_s:.1f}s · ~{rr.total_tokens_est} tokens · iterations={rr.iterations}')

## Step 3 — run a few more

Pick any questions that interest you. The engine will route each to the `papers` domain preset.

In [ ]:
QUESTIONS = [
    'What is FLARE active retrieval (Jiang et al. 2023) and what benchmark did they report?',
    'What does Reciprocal Rank Fusion do, and why is it useful for hybrid retrieval?',
    'What is the cross-encoder rerank model BAAI/bge-reranker-v2-m3 good at?',
]

for q in QUESTIONS:
    print('Q:', q)
    rr = run_query(q, domain='papers')
    print('A:', rr.answer)
    print(format_verified_summary(rr), f'· wall {rr.total_latency_s:.1f}s')
    print('-' * 80)

## What to try next

- Change `MODEL_SYNTHESIZER` to `llama-3.1-8b-instant` and compare answer quality vs the 70B model.
- Switch `domain` between `general`, `medical`, `papers`, and see how the synthesize prompt changes.
- Flip `ENABLE_FETCH=1` if `SEARXNG_URL` points at a real public instance — you'll get full-page trafilatura extractions in the trace.

Next tutorial: **03 — Build your own corpus** (upload PDFs, query them locally).